In [1]:
import os.path as osp

import torch
import torch.nn.functional as F

import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import SplineConv
from torch_geometric.typing import WITH_TORCH_SPLINE_CONV

if not WITH_TORCH_SPLINE_CONV:
    quit("This example requires 'torch-spline-conv'")

In [2]:
dataset = 'Cora'
transform = T.Compose([
    T.RandomNodeSplit(num_val=500, num_test=500),
    T.TargetIndegree(),
])
# path = osp.join(osp.dirname(osp.realpath(__file__)), '..', 'data', dataset)
dataset = Planetoid('data/Planetoid', dataset, transform=transform)
data = dataset[0]

In [3]:
class Net(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = SplineConv(dataset.num_features, 16, dim=1, kernel_size=2)
        self.conv2 = SplineConv(16, dataset.num_classes, dim=1, kernel_size=2)

    def forward(self):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        x = F.dropout(x, training=self.training)
        x = F.elu(self.conv1(x, edge_index, edge_attr))
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index, edge_attr)
        return F.log_softmax(x, dim=1)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, data = Net().to(device), data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-3)


def train():
    model.train()
    optimizer.zero_grad()
    F.nll_loss(model()[data.train_mask], data.y[data.train_mask]).backward()
    optimizer.step()


@torch.no_grad()
def test():
    model.eval()
    log_probs, accs = model(), []
    for _, mask in data('train_mask', 'test_mask'):
        pred = log_probs[mask].max(1)[1]
        acc = pred.eq(data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs


for epoch in range(1, 201):
    train()
    train_acc, test_acc = test()
    print(f'Epoch: {epoch:03d}, Train: {train_acc:.4f}, Test: {test_acc:.4f}')

/home/lubojjan/micromamba/envs/diploma/lib/python3.12/site-packages/torch_geometric/nn/conv/spline_conv.py:133: UserWarning: We do not recommend using the non-optimized CPU version of `SplineConv`. If possible, please move your data to GPU.
  warnings.warn(


Epoch: 001, Train: 0.3431, Test: 0.3240
Epoch: 002, Train: 0.3964, Test: 0.3560
Epoch: 003, Train: 0.5111, Test: 0.4380
Epoch: 004, Train: 0.6581, Test: 0.5840
Epoch: 005, Train: 0.7904, Test: 0.7160
Epoch: 006, Train: 0.8648, Test: 0.8080
Epoch: 007, Train: 0.9005, Test: 0.8340
Epoch: 008, Train: 0.9110, Test: 0.8600
Epoch: 009, Train: 0.9157, Test: 0.8660
Epoch: 010, Train: 0.9186, Test: 0.8700
Epoch: 011, Train: 0.9192, Test: 0.8700
Epoch: 012, Train: 0.9215, Test: 0.8660
Epoch: 013, Train: 0.9233, Test: 0.8620
Epoch: 014, Train: 0.9268, Test: 0.8660
Epoch: 015, Train: 0.9274, Test: 0.8720
Epoch: 016, Train: 0.9327, Test: 0.8740
Epoch: 017, Train: 0.9362, Test: 0.8800
Epoch: 018, Train: 0.9362, Test: 0.8800
Epoch: 019, Train: 0.9403, Test: 0.8840
Epoch: 020, Train: 0.9438, Test: 0.8860
Epoch: 021, Train: 0.9450, Test: 0.8800


KeyboardInterrupt: 